In [1]:
from datasets import load_dataset, load_from_disk, Dataset
from modelscope.models.multi_modal.mplug.clip import load_from_config
from modelscope.pipelines.nlp.document_grounded_dialog_rerank_pipeline import load_data

from chapter_8.classification_demo import trainloader

D:\PycharmProjects\llm_sft\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
datasets = load_dataset("madao33/new-title-chinese")
datasets

Generating validation split: 100%|██████████| 1679/1679 [00:00<00:00, 11014.94 examples/s]


DatasetDict({
    train: Dataset({
        features: ['title', 'content'],
        num_rows: 5850
    })
    validation: Dataset({
        features: ['title', 'content'],
        num_rows: 1679
    })
})

In [4]:
datasets = load_dataset("madao33/new-title-chinese", split="train[:10]")
datasets

Dataset({
    features: ['title', 'content'],
    num_rows: 10
})

In [5]:
datasets = load_dataset("madao33/new-title-chinese")
datasets

DatasetDict({
    train: Dataset({
        features: ['title', 'content'],
        num_rows: 5850
    })
    validation: Dataset({
        features: ['title', 'content'],
        num_rows: 1679
    })
})

In [6]:
datasets["train"][0]


{'title': '望海楼美国打“台湾牌”是危险的赌博',
 'content': '近期，美国国会众院通过法案，重申美国对台湾的承诺。对此，中国外交部发言人表示，有关法案严重违反一个中国原则和中美三个联合公报规定，粗暴干涉中国内政，中方对此坚决反对并已向美方提出严正交涉。\n事实上，中国高度关注美国国内打“台湾牌”、挑战一中原则的危险动向。近年来，作为“亲台”势力大本营的美国国会动作不断，先后通过“与台湾交往法”“亚洲再保证倡议法”等一系列“挺台”法案，“2019财年国防授权法案”也多处触及台湾问题。今年3月，美参院亲台议员再抛“台湾保证法”草案。众院议员继而在4月提出众院版的草案并在近期通过。上述法案的核心目标是强化美台关系，并将台作为美“印太战略”的重要伙伴。同时，“亲台”议员还有意制造事端。今年2月，5名共和党参议员致信众议院议长，促其邀请台湾地区领导人在国会上发表讲话。这一动议显然有悖于美国与台湾的非官方关系，其用心是实质性改变美台关系定位。\n上述动向出现并非偶然。在中美建交40周年之际，两国关系摩擦加剧，所谓“中国威胁论”再次沉渣泛起。美国对华认知出现严重偏差，对华政策中负面因素上升，保守人士甚至成立了“当前中国威胁委员会”。在此背景下，美国将台海关系作为战略抓手，通过打“台湾牌”在双边关系中增加筹码。特朗普就任后，国会对总统外交政策的约束力和塑造力加强。其实国会推动通过涉台法案对行政部门不具约束力，美政府在2018年并未提升美台官员互访级别，美军舰也没有“访问”台湾港口，保持着某种克制。但从美总统签署国会通过的法案可以看出，国会对外交产生了影响。立法也为政府对台政策提供更大空间。\n然而，美国需要认真衡量打“台湾牌”成本。首先是美国应对危机的代价。美方官员和学者已明确发出警告，美国卷入台湾问题得不偿失。美国学者曾在媒体发文指出，如果台海爆发危机，美国可能需要“援助”台湾，进而导致新的冷战乃至与中国大陆的冲突。但如果美国让台湾自己面对，则有损美国的信誉，影响美盟友对同盟关系的支持。其次是对中美关系的危害。历史证明，中美合则两利、斗则两伤。中美关系是当今世界最重要的双边关系之一，保持中美关系的稳定发展，不仅符合两国和两国人民的根本利益，也是国际社会的普遍期待。美国蓄意挑战台湾问题的底线，加剧中美关系的复杂性和不确定性，损害两国在重要领域合作，损人又害己

In [7]:
datasets["train"].column_names

['title', 'content']

In [8]:
datasets["train"].select([0, 1, 2])


Dataset({
    features: ['title', 'content'],
    num_rows: 3
})

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")


def prepare_dataset(example):
    model_inputs = tokenizer(example["content"], max_length=512, truncation=True)
    labels = tokenizer(example["title"], max_length=32, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [11]:
processed_datasets = datasets.map(prepare_dataset)
processed_datasets

Map: 100%|██████████| 1679/1679 [00:08<00:00, 208.89 examples/s]


DatasetDict({
    train: Dataset({
        features: ['title', 'content', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5850
    })
    validation: Dataset({
        features: ['title', 'content', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1679
    })
})

In [17]:
processed_datasets = datasets.map(prepare_dataset, batched=True)
processed_datasets

DatasetDict({
    train: Dataset({
        features: ['title', 'content', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5850
    })
    validation: Dataset({
        features: ['title', 'content', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1679
    })
})

In [18]:
processed_datasets.save_to_disk("./data/new-title-chinese")

Saving the dataset (1/1 shards): 100%|██████████| 1679/1679 [00:00<00:00, 88833.00 examples/s]


In [23]:
from datasets import load_from_disk

processed_datasets = load_from_disk("./data/new-title-chinese")
processed_datasets

DatasetDict({
    train: Dataset({
        features: ['title', 'content', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5850
    })
    validation: Dataset({
        features: ['title', 'content', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1679
    })
})

In [24]:
dataset = load_dataset("csv", data_files="./ChnSentiCorp_htl_all.csv")
dataset


Generating train split: 7766 examples [00:00, 73158.87 examples/s]


DatasetDict({
    train: Dataset({
        features: ['label', 'review'],
        num_rows: 7766
    })
})

In [28]:
from datasets import Dataset

dataset = Dataset.from_csv("./ChnSentiCorp_htl_all.csv")
dataset

Dataset({
    features: ['label', 'review'],
    num_rows: 7766
})

In [29]:
import pandas as pd


In [30]:
data = pd.read_csv("./ChnSentiCorp_htl_all.csv")
data.head()

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"


In [31]:
dataset = Dataset.from_pandas(data)
dataset

Dataset({
    features: ['label', 'review'],
    num_rows: 7766
})

In [32]:


from transformers import DataCollatorWithPadding

In [33]:
dataset = load_dataset("csv", data_files="./ChnSentiCorp_htl_all.csv", split="train")
dataset = dataset.filter(lambda x: x["review"] is not None)
dataset


Generating train split: 7766 examples [00:00, 94648.41 examples/s]
Filter: 100%|██████████| 7766/7766 [00:00<00:00, 151845.40 examples/s]


Dataset({
    features: ['label', 'review'],
    num_rows: 7765
})

In [34]:
def process_func(example):
    model_inputs = tokenizer(example["review"], max_length=128, truncation=True)
    model_inputs["labels"] = example["label"]
    return model_inputs

In [35]:
tokenized_dataset = dataset.map(process_func, batched=True, remove_columns=dataset.column_names)
tokenized_dataset

Map: 100%|██████████| 7765/7765 [00:04<00:00, 1728.46 examples/s]


Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 7765
})

In [36]:
print(tokenized_dataset[:3])

{'input_ids': [[101, 6655, 4895, 2335, 3763, 1062, 6662, 6772, 6818, 117, 852, 3221, 1062, 769, 2900, 4850, 679, 2190, 117, 1963, 3362, 3221, 107, 5918, 7355, 5296, 107, 4638, 6413, 117, 833, 7478, 2382, 7937, 4172, 119, 2456, 6379, 4500, 1166, 4638, 6662, 5296, 119, 2791, 7313, 6772, 711, 5042, 1296, 119, 102], [101, 1555, 1218, 1920, 2414, 2791, 8024, 2791, 7313, 2523, 1920, 8024, 2414, 3300, 100, 2160, 8024, 3146, 860, 2697, 6230, 5307, 3845, 2141, 2669, 679, 7231, 106, 102], [101, 3193, 7623, 1922, 2345, 8024, 3187, 6389, 1343, 1914, 2208, 782, 8024, 6929, 6804, 738, 679, 1217, 7608, 1501, 4638, 511, 6983, 2421, 2418, 6421, 7028, 6228, 671, 678, 6821, 702, 7309, 7579, 749, 511, 2791, 7313, 3315, 6716, 2523, 1962, 511, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [38]:
collatro = DataCollatorWithPadding(tokenizer=tokenizer)

In [39]:
from torch.utils.data import DataLoader


In [42]:
dl = DataLoader(tokenized_dataset, batch_size=8, collate_fn=collatro, shuffle=True)

In [44]:
next(enumerate(dl))

(0,
 {'input_ids': tensor([[ 101, 2769, 2697,  ...,    0,    0,    0],
         [ 101, 6983, 2421,  ...,    0,    0,    0],
         [ 101, 7676, 3949,  ...,    0,    0,    0],
         ...,
         [ 101, 3238, 5831,  ...,    0,    0,    0],
         [ 101, 2769,  702,  ..., 1373, 7008,  102],
         [ 101, 2791, 7313,  ...,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([1, 1, 1, 1, 1, 1, 0, 1])})

In [50]:
num = 0
for batch in dl:
    print(batch["input_ids"].size())

    num += 1
    if num > 1000:
        break

torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 113])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8

In [51]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [52]:
from datasets import load_dataset

In [61]:
datasets = load_dataset("csv", data_files="./ChnSentiCorp_htl_all.csv", split="train")
datasets = datasets.filter(lambda x: x["review"] is not None)
datasets

Dataset({
    features: ['label', 'review'],
    num_rows: 7765
})

In [62]:
datasets = datasets.train_test_split(test_size=0.1)
datasets


DatasetDict({
    train: Dataset({
        features: ['label', 'review'],
        num_rows: 6988
    })
    test: Dataset({
        features: ['label', 'review'],
        num_rows: 777
    })
})

In [63]:
import torch

tokenizer = AutoTokenizer.from_pretrained("hfl/rbt3")


In [64]:
def process_func(example):
    model_inputs = tokenizer(example["review"], max_length=128, truncation=True)
    model_inputs["labels"] = example["label"]
    return model_inputs

In [66]:
tokenized_dataset = datasets.map(process_func, batched=True, remove_columns=datasets["train"].column_names)

Map: 100%|██████████| 777/777 [00:00<00:00, 2120.49 examples/s]


In [67]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 6988
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 777
    })
})

In [68]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

trainloader = DataLoader(tokenized_dataset["train"], batch_size=32,
                         collate_fn=DataCollatorWithPadding(tokenizer=tokenizer), shuffle=True)
validloader = DataLoader(tokenized_dataset["test"], batch_size=64,
                         collate_fn=DataCollatorWithPadding(tokenizer=tokenizer), shuffle=True)


In [69]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('hfl/rbt3')
if torch.cuda.is_available():
    model = model.cuda()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at hfl/rbt3 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [70]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [71]:
def evaluate():
    model.eval()
    acc_num = 0
    with torch.inference_mode():
        for batch in validloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            output = model(**batch)
            pred = torch.argmax(output.logits, dim=-1)
            acc_num += (pred.long() == batch['labels'].long()).float().sum()
    return acc_num / len(validloader.dataset)


def train(epoch=3, log_step=100):
    global_step = 0
    for ep in range(epoch):
        model.train()
        for batch in trainloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            optimizer.zero_grad()
            output = model(**batch)
            output.loss.backward()
            optimizer.step()
            if global_step % log_step == 0:
                print(f"ep: {ep}, global_step: {global_step}, loss: {output.loss.item()}")
            global_step += 1
        acc = evaluate()
        print(f"ep: {ep}, global_step: {global_step}, acc: {acc}")


In [73]:
train()

ep: 0, global_step: 0, loss: 0.16315442323684692
ep: 0, global_step: 100, loss: 0.1540452539920807
ep: 0, global_step: 200, loss: 0.25825417041778564
ep: 0, global_step: 219, acc: 0.8906049132347107
ep: 1, global_step: 300, loss: 0.08283618837594986
ep: 1, global_step: 400, loss: 0.15396465361118317
ep: 1, global_step: 438, acc: 0.8687258958816528
ep: 2, global_step: 500, loss: 0.04673690348863602
ep: 2, global_step: 600, loss: 0.014193225651979446
ep: 2, global_step: 657, acc: 0.8803088665008545
